
# 01 – Data Preparation & Transformation (Student Performance)
**Arabic + English walkthrough** for tasks **A & B** from your project brief.

> **Scope**
> - Load `student-mat.csv` (or `student-por.csv`)
> - Validate schema & types, check duplicates/missing/outliers
> - Produce a short **Data Quality Report**
> - Transformations: encoding, scaling
> - Feature engineering & *leakage note* variants

> **How to use this notebook**
> 1) Adjust `DATA_PATH` to your local file (Windows path provided by you).
> 2) Run cells top-to-bottom.
> 3) Generated artifacts (clean CSV & report) will be saved under `outputs/`.


In [1]:

# ==============================
# CONFIG
# ==============================
from pathlib import Path

# ⚠️ Set your local path to the CSV (Windows path you shared)
DATA_PATH = Path(r"C:\Users\carnival\Downloads\student-mat.csv")  # change if needed

# Separator is ';' for UCI Student dataset
CSV_SEP = ';'

# Output directory for artifacts
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Using file:", DATA_PATH.resolve())


Using file: C:\Users\carnival\Downloads\student-mat.csv


In [2]:

# ==============================
# IMPORTS
# ==============================
import pandas as pd
import numpy as np
import json, textwrap, os, sys
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [3]:

# ==============================
# LOAD DATA
# ==============================
# Note: UCI files use ';' as separator
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)
print("Shape:", df.shape)
df.head()


Shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10



## A) Schema & Types Validation
- Validate that all **expected columns** exist.
- Show data types and basic info.
- Check duplicates and missing values.


In [4]:

# Common expected columns for Student Performance dataset (Math/Portuguese)
expected_cols = [
    'school','sex','age','address','famsize','Pstatus','Medu','Fedu','Mjob','Fjob',
    'reason','guardian','traveltime','studytime','failures','schoolsup','famsup','paid',
    'activities','nursery','higher','internet','romantic','famrel','freetime','goout',
    'Dalc','Walc','health','absences','G1','G2','G3'
]

present = [c for c in expected_cols if c in df.columns]
missing = [c for c in expected_cols if c not in df.columns]
extras  = [c for c in df.columns if c not in expected_cols]

print("Present columns:", len(present))
print("Missing columns:", missing)
print("Extra columns (not in expected list):", extras)

print("\nData types:")
display(df.dtypes)

print("\nBasic info:")
df.info()


Present columns: 33
Missing columns: []
Extra columns (not in expected list): []

Data types:


school        object
sex           object
age            int64
address       object
famsize       object
Pstatus       object
Medu           int64
Fedu           int64
Mjob          object
Fjob          object
reason        object
guardian      object
traveltime     int64
studytime      int64
failures       int64
schoolsup     object
famsup        object
paid          object
activities    object
nursery       object
higher        object
internet      object
romantic      object
famrel         int64
freetime       int64
goout          int64
Dalc           int64
Walc           int64
health         int64
absences       int64
G1             int64
G2             int64
G3             int64
dtype: object


Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 2

In [5]:

# ==============================
# DUPLICATES & MISSING VALUES
# ==============================
n_dupes = df.duplicated().sum()
missing_counts = df.isna().sum().sort_values(ascending=False)

print(f"Duplicate rows: {n_dupes}")
print("\nMissing values per column:")
display(missing_counts[missing_counts>0])

# Decide: drop duplicates if any
if n_dupes > 0:
    df = df.drop_duplicates().copy()
    print(f"Dropped {n_dupes} duplicate rows. New shape: {df.shape}")
else:
    print("No duplicate rows detected.")


Duplicate rows: 0

Missing values per column:


Series([], dtype: int64)

No duplicate rows detected.


In [6]:

# ==============================
# OUTLIERS (Numeric columns) – IQR method
# ==============================
num_cols = df.select_dtypes(include=np.number).columns.tolist()

def iqr_outlier_counts(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k*iqr, q3 + k*iqr
    return int(((s < lower) | (s > upper)).sum())

outlier_summary = {col: iqr_outlier_counts(df[col].dropna()) for col in num_cols}
outlier_summary = pd.Series(outlier_summary).sort_values(ascending=False)
print("Outlier counts by numeric column (IQR, k=1.5):")
display(outlier_summary)


Outlier counts by numeric column (IQR, k=1.5):


failures      83
studytime     27
famrel        26
freetime      19
Dalc          18
absences      15
G2            13
traveltime     8
Fedu           2
age            1
Medu           0
goout          0
Walc           0
health         0
G1             0
G3             0
dtype: int64


### Data Quality Report (Auto-generated)
This section creates a concise **Markdown report** summarizing:
- Shape, duplicates, missing values
- Outlier counts for numeric columns
- Schema check (present/missing/extra columns)


In [7]:

report = []
report.append(f"# Data Quality Report – Student Performance\n")
report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
report.append(f"\n## Shape\n- Rows: {df.shape[0]}\n- Columns: {df.shape[1]}\n")

report.append("## Duplicates\n")
report.append(f"- Duplicate rows: {n_dupes}\n")

miss_any = missing_counts[missing_counts>0]
report.append("## Missing Values\n")
if miss_any.empty:
    report.append("- No missing values detected.\n")
else:
    report.append(miss_any.to_frame('missing').to_markdown() + "\n")

report.append("## Outliers (IQR)\n")
report.append(pd.Series(outlier_summary, name='outliers').to_frame().to_markdown() + "\n")

report.append("## Schema Check\n")
report.append(f"- Present columns ({len(present)}): {present}\n")
report.append(f"- Missing columns ({len(missing)}): {missing}\n")
report.append(f"- Extra columns ({len(extras)}): {extras}\n")

# Save report
report_path = OUTPUT_DIR / "data_quality_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report))

print("Report saved to:", report_path.resolve())


Report saved to: C:\Users\carnival\Downloads\outputs\data_quality_report.md



## Cleaning Decisions
- **Duplicates**: dropped if any found.  
- **Missing values**: dataset is typically complete; if any appear, consider:
  - Categorical → impute with mode
  - Numeric → impute with median
- **Outliers**: flagged, not automatically removed (they may be informative, e.g., absences).

You can customize the rules below if your run shows issues.


In [8]:

# ==============================
# OPTIONAL: Simple Imputation (only if missing exist)
# ==============================
from sklearn.impute import SimpleImputer

if (df.isna().sum() > 0).any():
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    num_cols = df.select_dtypes(include=np.number).columns.tolist()

    if cat_cols:
        imp_mode = SimpleImputer(strategy='most_frequent')
        df[cat_cols] = imp_mode.fit_transform(df[cat_cols])
    if num_cols:
        imp_med = SimpleImputer(strategy='median')
        df[num_cols] = imp_med.fit_transform(df[num_cols])

    print("Imputation done (mode for categoricals, median for numerics).")
else:
    print("No imputation needed.")


No imputation needed.



## B) Transformations & Feature Engineering
- **Encoding**: one-hot encode categoricals (school, sex, address, Mjob, ...)
- **Scaling**: standardize numeric features (for ML & K-Means)
- **Feature Engineering examples**:
  - `attendance_proxy` from `absences` (higher = worse attendance)
  - `avg_G` = average of G1–G3
  - Binary target `pass` = 1 if G3 ≥ 10 else 0
- **Leakage Note**: Create two variants for modeling later:
  - **Variant A (with G1 & G2)** when predicting `G3`
  - **Variant B (without G1 & G2)** (more realistic early signal)


In [9]:

# Identify categorical vs numeric
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=np.number).columns.tolist()

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# Feature engineering
df['attendance_proxy'] = df['absences']  # simple proxy: more absences -> weaker attendance
df['avg_G'] = df[['G1','G2','G3']].mean(axis=1)

# Binary pass/fail target based on G3 >= 10 (Portuguese grading 0–20)
df['pass'] = (df['G3'] >= 10).astype(int)

# One-hot encoding for categoricals
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("Encoded shape:", df_enc.shape)
df_enc.head()


Categorical columns: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Numeric columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
Encoded shape: (395, 45)


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,attendance_proxy,avg_G,pass,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,1,3,6,5,6,6,6,5.666667,0,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,True,True,False,False
1,17,1,1,1,2,0,5,3,3,1,1,3,4,5,5,6,4,5.333333,0,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False
2,15,1,1,1,2,3,4,3,2,2,3,3,10,7,8,10,10,8.333333,1,False,False,True,True,True,False,False,False,False,False,True,False,False,False,True,False,True,False,True,False,True,False,True,True,True,False
3,15,4,2,1,3,0,3,2,2,1,1,5,2,15,14,15,2,14.666667,1,False,False,True,False,True,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,True,True,True,True,True,True
4,16,3,3,1,2,0,4,3,2,1,2,5,4,6,10,10,4,8.666667,1,False,False,True,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,True,True,False,True,True,False,False


In [10]:

from sklearn.preprocessing import StandardScaler

# We'll scale numeric features for algorithms sensitive to scale (e.g., K-Means, SVM)
scale_cols = [c for c in df_enc.columns if c in num_cols or c in ['attendance_proxy','avg_G']]
scaler = StandardScaler()
df_enc[scale_cols] = scaler.fit_transform(df_enc[scale_cols])

print(f"Scaled {len(scale_cols)} numeric features.")
df_enc.head()


Scaled 18 numeric features.


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,attendance_proxy,avg_G,pass,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,1.023046,1.143856,1.360371,0.792251,-0.042286,-0.449944,0.062194,-0.236010,0.801479,-0.540699,-1.003789,-0.399289,0.036424,-1.782467,-1.254791,-0.964934,0.036424,-1.357670,0,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,True,True,False,False
1,0.238380,-1.600009,-1.399970,-0.643249,-0.042286,-0.449944,1.178860,-0.236010,-0.097908,-0.540699,-1.003789,-0.399289,-0.213796,-1.782467,-1.520979,-0.964934,-0.213796,-1.447953,0,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,True,False
2,-1.330954,-1.600009,-1.399970,-0.643249,-0.042286,3.589323,0.062194,-0.236010,-0.997295,0.583385,0.551100,-0.399289,0.536865,-1.179147,-0.722415,-0.090739,0.536865,-0.635408,1,False,False,True,True,True,False,False,False,False,False,True,False,False,False,True,False,True,False,True,False,True,False,True,True,True,False
3,-1.330954,1.143856,-0.479857,-0.643249,1.150779,-0.449944,-1.054472,-1.238419,-0.997295,-0.540699,-1.003789,1.041070,-0.464016,1.234133,0.874715,1.002004,-0.464016,1.079965,1,False,False,True,False,True,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,True,True,True,True,True,True
4,-0.546287,0.229234,0.440257,-0.643249,-0.042286,-0.449944,0.062194,-0.236010,-0.997295,-0.540699,-0.226345,1.041070,-0.213796,-1.480807,-0.190038,-0.090739,-0.213796,-0.545125,1,False,False,True,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,True,True,False,True,True,False,False


In [11]:

# Save cleaned & transformed data
clean_path = OUTPUT_DIR / "student_mat_clean.csv"
enc_path   = OUTPUT_DIR / "student_mat_encoded_scaled.csv"

df.to_csv(clean_path, index=False)
df_enc.to_csv(enc_path, index=False)

print("Saved:")
print("-", clean_path.resolve())
print("-", enc_path.resolve())


Saved:
- C:\Users\carnival\Downloads\outputs\student_mat_clean.csv
- C:\Users\carnival\Downloads\outputs\student_mat_encoded_scaled.csv


In [12]:

# Prepare columns for leakage variants when predicting G3 later:
# Variant A: include G1 & G2
variant_A_cols = [c for c in df_enc.columns if c != 'pass']  # keep all (pass kept for classification target usage)
# Variant B: drop G1 & G2 features
variant_B_cols = [c for c in df_enc.columns if not c.startswith('G1') and not c.startswith('G2')]

# Save column lists as JSON for reuse in later notebooks
cfg = {
    "variant_A_cols": variant_A_cols,
    "variant_B_cols": variant_B_cols,
    "target_binary": "pass",
    "target_regression": "G3",
    "scale_cols": scale_cols
}
cfg_path = OUTPUT_DIR / "feature_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("Feature config saved to:", cfg_path.resolve())


Feature config saved to: C:\Users\carnival\Downloads\outputs\feature_config.json
